# The CARLA campaign

What the synthetic map established, demonstrated in a real town. Per the
2026-08-10 split, **CARLA's job is to show the result carries over, not to
re-establish it** — the 5% CI bar lives on the synthetic map, where a trial
costs seconds instead of minutes. So this runs enough seeds to demonstrate
transfer, and no more.

## Why the earlier CARLA runs were purged

Every CARLA run before 2026-08-11 was made against a travel model that
estimated `length / 30 km/h` for the whole town, which measured **1.75x
optimistic** — the policy planned pickup windows in a time base the bus
could not live in. Two things were wrong and both are now fixed:

- **The speed limits were readable all along.** Town01 publishes 22
  type-274 landmarks at 30/60/90, and a driving vehicle reads them back.
  Its arterials — 37.7% of the town's length — were priced at 30.
- **What a bus achieves is not what the sign says.** 32% of every run is
  spent stationary at a red light. That is a bias, not noise (execution
  noise measures **0.034**), so it belongs in the estimate:
  `traffic_efficiency`, calibrated per town by `src/map/carla_calibrate.py`.

After both, the model predicts Town01's legs with **bias 1.000x** and
residual sigma **0.278**, down from 1.75x and 0.408.

**So these results are not comparable with anything recorded before that
date**, which is why the old runs are gone rather than merely superseded.

## What this campaign asks

1. **§3.2.2** — fulfilment >= 80%, and a positive sum of percentage
   differences against a fixed-route bus. The headline claim.
2. **Does L1's ride advantage survive real driving?** The synthetic
   campaign puts it at -111.5 s pooled over 25 maps, and the execution-noise
   study predicted it *widens* under unreliable driving.
3. **Does the calibration hold when the policy picks the legs?** It was
   measured on a fixed tour of every stop. A campaign drives whatever the
   policy chooses, which is a different and more demanding sample.
4. **Is the service window actually kept?** Newly worth asking. Against a
   2.2x optimistic clock a breach was near-guaranteed and told us nothing
   about the policy; against a calibrated one it is a real measurement.

## Running it

Needs a live server: `docker compose up -d`, and give it a minute. It is
**sequential** — `CarlaBackend` sets `parallel_safe = False`, one server
cannot host a campaign per core. Expect roughly three minutes per trial.

Everything lands in the results store as it completes, so **stopping and
re-running continues rather than restarting**. A trial that raises is caught
per seed here, so one bad trial costs one trial rather than the campaign.

## Configuration

In [ ]:
TOWN = "Town01"
NPC_VEHICLES = 20          # traffic costs ~3x wall clock and changes service
                           # not at all, but it is what the noise study models
SEEDS = list(range(10))    # demonstration of transfer, not a 5% CI

# Matches the sizing pilot, which priced a trial here at roughly 3 minutes.
MAX_REQUESTS = 12
DEMAND_INTERVAL_S = 150.0
CAPACITY = 20

WINDOW = 900.0             # the service window the policies plan against
PATIENCE = 1800.0          # when a waiting passenger gives up. Deliberately
                           # LOOSER than the window, so a breach is possible:
                           # at window == patience nobody who breaches is ever
                           # fulfilled, and the breach rate is a flat zero.
DWELL_S = 10.0

FIXED_DELTA_S = 0.05       # never above 0.1; past CARLA's substepping limit
                           # the physics silently stops keeping up
MAX_TIME = 10000.0         # a ceiling that truncates a run looks exactly like
                           # a coverage failure, so it is set well clear

CARLA_HOST, CARLA_PORT = "localhost", 2000

In [ ]:
import math
import statistics
import time

import matplotlib.pyplot as plt
import pandas as pd
from cycler import cycler
from scipy import stats

from src.algorithm.FixedRoutePolicy import FixedRoutePolicy
from src.algorithm.ReorderingInsertionPolicy import ReorderingInsertionPolicy
from src.algorithm.WindowInsertionPolicy import WindowInsertionPolicy
from src.experiment import run_campaign
from src.experiment.store import (
    DEFAULT_RUNS_DIR,
    load_runs,
    to_results,
)
from src.experiment.summary import summarize_results
from src.map.CarlaMapProvider import CarlaMapProvider
from src.simulation.CarlaBackend import CarlaBackend, stop_positions_from

from IPython.display import Markdown, display

# Validated palette slots 1-3 (all-pairs CVD 9.2, normal-vision 27.6 on this
# surface). Every chart here ships with its table beside it, which is the
# relief the green slot's sub-3:1 contrast requires.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
SURFACE, INK, INK_SOFT, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e1"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "axes.prop_cycle": cycler(color=SERIES),
    "axes.edgecolor": INK_SOFT, "axes.labelcolor": INK_SOFT,
    "axes.titlecolor": INK, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.labelsize": 9, "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": GRID, "grid.linewidth": 0.8, "xtick.color": INK_SOFT,
    "ytick.color": INK_SOFT, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "legend.frameon": False, "legend.fontsize": 9, "figure.dpi": 110,
})

POLICY_LABELS = ["Fixed route", "Window W900", "Reordering W900"]
POLICIES = [
    ("Fixed route", lambda g: FixedRoutePolicy(g)),
    ("Window W900", lambda g: WindowInsertionPolicy(g, max_wait_s=WINDOW)),
    ("Reordering W900",
     lambda g: ReorderingInsertionPolicy(g, max_wait_s=WINDOW)),
]

## Preflight

Four things that have each silently invalidated a run before. Read the
output; do not skip to the campaign.

In [ ]:
provider = CarlaMapProvider(TOWN)
positions = stop_positions_from(provider)
problems = provider.validate()
efficiency = provider.traffic_efficiency()

print(f"{TOWN}: {len(provider.get_road_edges())} edges, "
      f"{len(provider.get_stops())} stops")
print("map health:", "OK" if not problems else problems)
print(f"traffic_efficiency: {efficiency}")

if efficiency == 1.0:
    display(Markdown(
        "> **STOP — this town is uncalibrated.** `traffic_efficiency` of 1.0 "
        "means the model assumes the bus achieves the posted limit exactly, "
        "which measured **1.75x optimistic** on Town01. Run\n"
        "> ```\n> ./run.sh -m src.map.carla_calibrate --town "
        f"{TOWN} --write\n> ```\n"
        "> against a live server first. Results produced now would repeat "
        "the error the old runs were purged for."))

limits = sorted(set(provider._topology.speed_limit_overrides.values()))
print(f"per-road speed limits read from signs: {limits} km/h "
      f"({len(provider._topology.speed_limit_overrides)} roads), "
      f"default {provider._topology.default_speed_limit_kph} km/h")

# The window must be tighter than patience or the breach rate is
# structurally zero -- anyone who waits past the window is already cancelled.
assert WINDOW < PATIENCE, "breach rate cannot be measured at window >= patience"
print(f"window {WINDOW:.0f}s < patience {PATIENCE:.0f}s, so a breach is "
      f"observable in the {PATIENCE - WINDOW:.0f}s band between them")

In [ ]:
# The world identity. Everything that changes what the bus experiences must
# be in here: the store hashes it, and anything omitted silently merges two
# genuinely different runs. traffic_efficiency and the speed-limit rule are
# here because they change every estimate the policy makes -- omitting them
# would let a pre-calibration run answer for a post-calibration one.
WORLD = {
    "town": TOWN,
    "backend": "CarlaBackend",
    "npc_vehicles": NPC_VEHICLES,
    "fixed_delta_s": FIXED_DELTA_S,
    "no_rendering": True,
    "blueprint": "vehicle.volkswagen.t2_2021",
    "traffic_efficiency": efficiency,
    "speed_limits": "landmark-harmonic-v1",
}

TRIAL = dict(
    max_requests=MAX_REQUESTS, demand_interval_s=DEMAND_INTERVAL_S,
    capacity=CAPACITY, max_wait_s=PATIENCE, dwell_s=DWELL_S,
    max_time=MAX_TIME,
)


def backend_factory_for(seed: int):
    """
    Build this seed's world builder.
    :param seed: Trial seed; varies traffic and spawn placement with it.
    :return: A factory run_trial can call.
    """
    def build(graph, start_stop_id: str) -> CarlaBackend:
        """Build the CARLA backend for one trial."""
        return CarlaBackend(
            stop_positions=positions, position=start_stop_id, town=TOWN,
            host=CARLA_HOST, port=CARLA_PORT, fixed_delta_s=FIXED_DELTA_S,
            dwell_s=DWELL_S, npc_vehicles=NPC_VEHICLES, seed=seed,
            no_rendering=True,
        )
    return build


print(f"{len(POLICIES)} policies x {len(SEEDS)} seeds = "
      f"{len(POLICIES) * len(SEEDS)} trials, sequential, "
      f"~3 min each -> roughly "
      f"{len(POLICIES) * len(SEEDS) * 3 / 60:.1f} hours if none are cached")
display(Markdown("**World identity:** `" + str(WORLD) + "`"))

## Run it

One `run_campaign` call per seed rather than one for the lot, so a trial
that raises — a server hiccup, a spawn failure — costs that seed and not the
night. Everything already recorded is skipped, so re-running resumes.

In [ ]:
started = time.perf_counter()
ran = skipped = 0
failures = []

for seed in SEEDS:
    try:
        result = run_campaign(
            policies=POLICIES, seeds=[seed], world=WORLD,
            directory=DEFAULT_RUNS_DIR, provider=provider,
            backend_factory_for=backend_factory_for,
            n_jobs=1, progress=False, **TRIAL)
        ran += result.ran
        skipped += result.skipped
    except Exception as error:                    # noqa: BLE001
        # Deliberately broad: a CARLA failure can surface as almost
        # anything, and losing the remaining seeds to it would be worse.
        failures.append((seed, repr(error)))
        print(f"  seed {seed:<3} FAILED: {error!r}")
    print(f"  seed {seed:<3} ran={ran:<4} skipped={skipped:<4} "
          f"{(time.perf_counter() - started) / 60:6.1f} min")

print(f"\ndone -- {ran} run, {skipped} already recorded, "
      f"{len(failures)} seeds failed")
for seed, error in failures:
    print(f"  seed {seed}: {error}")

## Collect

In [ ]:
records = [
    r for r in load_runs(DEFAULT_RUNS_DIR)
    if r.identity.world.get("town") == TOWN
    and r.identity.world.get("backend") == "CarlaBackend"
    and r.identity.world.get("traffic_efficiency") == efficiency
    and r.identity.world.get("npc_vehicles") == NPC_VEHICLES
    and r.identity.trial.get("max_requests") == MAX_REQUESTS
    and r.identity.policy in POLICY_LABELS
]
print(f"{len(records)} runs")

rows = [{
    "policy": r.identity.policy, "seed": r.identity.seed,
    "served": r.kpi.served_fraction * 100,
    "wait": r.kpi.mean_wait_time, "ride": r.kpi.mean_ride_time,
    "throughput": r.kpi.throughput,
    "fulfilled": r.kpi.fulfilled_count,
    "breach": (100.0 * sum(w > WINDOW for w in r.wait_times)
               / len(r.wait_times) if r.wait_times else float("nan")),
    "sim_s": r.backend_stats.get("ticks", 0) * FIXED_DELTA_S,
    "distance_m": r.backend_stats.get("distance_m", 0.0),
    "legs": r.backend_stats.get("legs_started", 0),
    "red_s": r.backend_stats.get("red_light_ticks", 0) * FIXED_DELTA_S,
    "repaths": r.backend_stats.get("repaths", 0),
    "unroutable": r.backend_stats.get("unroutable_legs", 0),
} for r in records]
# Explicit columns so an empty store still yields a usable frame
# rather than a KeyError three cells later.
frame = pd.DataFrame(rows, columns=["policy", "seed", "served", "wait", "ride", "throughput",
     "fulfilled", "breach", "sim_s", "distance_m", "legs",
     "red_s", "repaths", "unroutable"])
HAVE_RUNS = not frame.empty
if not HAVE_RUNS:
    display(Markdown(
        '> **No runs yet.** Run the campaign cell above, or check that the\n'
        '> world identity here matches the one it filed under — a single\n'
        '> differing field files runs under an identity this filter misses.'))
if HAVE_RUNS:
    display(frame.groupby("policy")[["served", "wait", "ride", "throughput"]]
            .mean().round(1))

# A run that needed repaths or hit an unroutable leg drove a route nobody
# planned, so its KPIs describe something other than the policy.
suspect = frame[(frame["repaths"] > 0) | (frame["unroutable"] > 0)]
print(f"\nruns with repaths or unroutable legs: {len(suspect)}"
      + ("" if suspect.empty else " <- inspect before trusting these"))

## 1. §3.2.2 — the headline claim

Two criteria: fulfilment **>= 80%**, and a **positive sum** of percentage
differences against fixed route across wait, ride and throughput.

The gate is doing the real work. Wait and ride are computed over *fulfilled*
requests only, so a policy that abandons its hard riders scores well on
them — the sum must never be read on its own.

In [ ]:
if not HAVE_RUNS:
    display(Markdown('_Nothing to report until the campaign has run._'))
else:
    # paired=False: a seed missing for one policy should not drop it
    # from the others here, since this table is per-policy anyway.
    display(Markdown(summarize_results(
        to_results(records, paired=False)).markdown))

In [ ]:
if not HAVE_RUNS:
    display(Markdown('_Nothing to report until the campaign has run._'))
else:
    means = frame.groupby("policy")[["served", "wait", "ride", "throughput"]].mean()
    base = means.loc["Fixed route"]

    lines = ["| policy | fulfilment | wait Δ% | ride Δ% | throughput Δ% "
             "| **sum** | gate | §3.2.2 |", "|---|---|---|---|---|---|---|---|"]
    for label in POLICY_LABELS:
        if label == "Fixed route":
            continue
        row = means.loc[label]
        # Lower is better for wait and ride, higher for throughput.
        d_wait = 100 * (base["wait"] - row["wait"]) / base["wait"]
        d_ride = 100 * (base["ride"] - row["ride"]) / base["ride"]
        d_tput = 100 * (row["throughput"] - base["throughput"]) / base["throughput"]
        total = d_wait + d_ride + d_tput
        gate = row["served"] >= 80.0
        verdict = "**PASS**" if gate and total > 0 else "FAIL"
        lines.append(
            f"| {label} | {row['served']:.1f}% | {d_wait:+.1f} | {d_ride:+.1f} "
            f"| {d_tput:+.1f} | **{total:+.1f}** | "
            f"{'✓' if gate else '✗ <80%'} | {verdict} |")
    display(Markdown("\n".join(lines)))
    display(Markdown(
        f"Fixed route baseline: **{base['served']:.1f}%** served, "
        f"{base['wait']:.0f} s wait, {base['ride']:.0f} s ride, "
        f"{base['throughput']:.2f} fulfilled/min."))

## 2. Does L1's advantage survive real driving?

Paired by seed, since both policies met the same demand in the same town.
The synthetic campaign puts L1 − L0 mean ride at **−111.5 s** pooled over 25
maps, and the execution-noise study predicted the gap **widens** when the
driving is unreliable rather than exact.

In [ ]:
if not HAVE_RUNS:
    display(Markdown('_Nothing to report until the campaign has run._'))
else:
    l0 = frame[frame["policy"] == "Window W900"].set_index("seed")
    l1 = frame[frame["policy"] == "Reordering W900"].set_index("seed")
    shared = sorted(set(l0.index) & set(l1.index))
    print(f"{len(shared)} paired seeds")

    lines = ["| metric | Window W900 | Reordering W900 | L1 − L0 | p |",
             "|---|---|---|---|---|"]
    for column, unit in [("served", "%"), ("wait", " s"), ("ride", " s")]:
        a = l0.loc[shared, column].astype(float)
        b = l1.loc[shared, column].astype(float)
        delta = (b - a).mean()
        p = (stats.ttest_rel(b, a).pvalue if len(shared) > 1
             and (b - a).std() > 0 else float("nan"))
        lines.append(f"| mean {column} | {a.mean():.1f}{unit} | "
                     f"{b.mean():.1f}{unit} | **{delta:+.1f}{unit}** | "
                     f"{p:.4f} |")
    display(Markdown("\n".join(lines)))
    display(Markdown(
        f"Synthetic reference: L1 − L0 ride **−111.5 s** pooled over 25 maps "
        f"(p≈2e-138), **−70 s** on the campus map alone. With only "
        f"{len(shared)} seeds here, treat the sign and rough magnitude as the "
        f"claim and not the p-value."))

## 3. Did the calibration hold?

`traffic_efficiency` was measured on a fixed tour of every stop. A campaign
drives whatever the policy chooses, which is a different sample of the town —
so this is the check that the number transfers off its own tour.

The calibration measured **4.21 m/s** effective and **32.1%** of time at red
lights. If the campaign lands far from those, the factor is describing the
calibration tour rather than the town.

In [ ]:
if not HAVE_RUNS:
    display(Markdown('_Nothing to report until the campaign has run._'))
else:
    CALIBRATION_SPEED_MPS = 4.21
    CALIBRATION_RED_FRACTION = 0.321

    drive = frame[frame["sim_s"] > 0].copy()
    drive["speed"] = drive["distance_m"] / drive["sim_s"]
    drive["red_frac"] = drive["red_s"] / drive["sim_s"]

    display(drive.groupby("policy")[["speed", "red_frac", "legs", "sim_s"]]
            .mean().round(3))

    speed = drive["speed"].mean()
    red = drive["red_frac"].mean()
    display(Markdown(
        f"Campaign-wide **{speed:.2f} m/s** against the calibration's "
        f"{CALIBRATION_SPEED_MPS:.2f} "
        f"({100 * (speed / CALIBRATION_SPEED_MPS - 1):+.1f}%), and "
        f"**{100 * red:.1f}%** of time at red against "
        f"{100 * CALIBRATION_RED_FRACTION:.1f}%.\n\n"
        "Within a few percent means the factor describes the town. A large gap "
        "means the policy drives a systematically different part of it — which "
        "would be a finding about the calibration tour, not a failure of the "
        "campaign."))

## 4. Is the service window kept?

A **breach** is a fulfilled rider whose wait exceeded the window they were
accepted under. This is the first time the question is meaningful on CARLA:
against the old 2.2x optimistic clock the policy was promising windows it
had no way to keep, so a breach measured the model's error rather than the
policy's behaviour.

Synthetic reference at W750/900 s patience: **3.4%**, of which essentially
all is structural — the progress fallback, which drives to the nearest
pending pickup ignoring the window when nothing can be scheduled.

In [ ]:
if not HAVE_RUNS:
    display(Markdown('_Nothing to report until the campaign has run._'))
else:
    breaches = frame[frame["policy"] != "Fixed route"]
    figure, axis = plt.subplots(figsize=(7.0, 3.6))
    for colour, label in zip(SERIES[1:], POLICY_LABELS[1:]):
        subset = breaches[breaches["policy"] == label].sort_values("seed")
        axis.plot(subset["seed"], subset["breach"], marker="o", markersize=6,
                  lw=2.0, color=colour, label=label)
    axis.axhline(3.4, ls="--", lw=1.1, color=INK_SOFT)
    axis.text(0.02, 3.6, "synthetic reference 3.4%", color=INK_SOFT, fontsize=9,
              transform=axis.get_yaxis_transform())
    axis.set_xlabel("seed")
    axis.set_ylabel("% of fulfilled riders")
    axis.set_title("Promises broken, per seed")
    axis.grid(axis="y")
    axis.set_axisbelow(True)
    axis.legend()
    figure.tight_layout()
    plt.show()

    display(breaches.groupby("policy")[["breach", "served", "wait"]]
            .agg(["mean", "max"]).round(2))

## 5. What to conclude

Fill in once run. The four questions, and what each answer would mean:

1. **§3.2.2.** A pass here is the headline: the recommendation transfers from
   the synthetic map to a real town. Quote it as *demonstrated transfer at
   n=10 seeds*, never as an independent statistical result — the CI bar
   deliberately lives on the synthetic map.
2. **L1 vs L0.** The sign and rough magnitude are the claim. If the gap is
   wider than the synthetic −70 s, that is the execution-noise prediction
   coming true; if it is narrower or absent, say so plainly, because at ten
   seeds this is under-powered and a null here is weak evidence either way.
3. **The calibration.** Within a few percent means `traffic_efficiency`
   describes the town rather than its calibration tour, and that the
   per-town calibration recipe is sound for the maps still to come.
4. **The window.** Near the synthetic 3.4% would say the breach floor is
   structural and travels — worth stating, since it is the one number that
   would have been meaningless before the calibration.

Also worth recording either way: whether any run needed a repath or hit an
unroutable leg, since those describe driving nobody planned.